In [7]:
import sys
import socket
import importlib
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from pathlib import Path

In [8]:
sys.path.append(str(Path.cwd().parent))

In [9]:
from src.config import raw_data_dir, processed_data_path, rdp_epsilon, target_features

In [10]:

data_path = raw_data_dir / "sample_data.npy"
sim = np.load(data_path, allow_pickle=True)

df = pd.DataFrame(sim)

In [26]:
importlib.reload(sys.modules["src.preprocessing"])
from src.preprocessing import check_missing_values, drop_constant_columns, fit_preprocess_scalers, rdp

df_rdp = rdp(df, epsilon=rdp_epsilon)

np.save(processed_data_path, df_rdp.to_numpy())
print(f"Saved processed dataset to {processed_data_path}")


Saved processed dataset to /work/nvme/bhvr/cadence/Stars_VAE/data/processed/processed_data.npy


In [39]:
df_rdp.head()

,initial_mass,initial_z,star_age,mass,logR,logP,logRho,logT,luminosity,opacity,x_mass_fraction_H,y_mass_fraction_He,z_mass_fraction_metals,eps_nuc,eps_nuc_neu_total,eps_grav_nh,eps_grav,zone,q
0,16.5,0.02,1.124803e+07,15.672543,2.769037,2.632306,-8.726443,3.565941,56927.325018,0.001937,0.667269,0.319678,0.013054,0.042915,1.739978e-29,-0.003933,-2.621206,1.0,1.000000
1,16.5,0.02,1.124803e+07,15.672539,2.769035,2.632926,-8.725901,3.566019,56927.325024,0.001941,0.667269,0.319678,0.013054,0.042915,1.741227e-29,-0.003912,-2.614513,23.0,1.000000
2,16.5,0.02,1.124803e+07,15.672499,2.769014,2.639828,-8.719876,3.566898,56927.325077,0.001986,0.667269,0.319678,0.013054,0.042915,1.755225e-29,-0.003680,-2.540143,28.0,0.999997
3,16.5,0.02,1.124803e+07,15.672461,2.768993,2.647528,-8.713187,3.567908,56927.325127,0.002039,0.667269,0.319678,0.013054,0.042915,1.770999e-29,-0.003432,-2.457576,29.0,0.999995
4,16.5,0.02,1.124803e+07,15.672194,2.768858,2.680639,-8.684855,3.572687,56927.325445,0.002294,0.667269,0.319678,0.013054,0.042915,1.840559e-29,-0.002499,-2.110252,31.0,0.999978


In [28]:
check_missing_values(df_rdp)

,missing_values,null_percent


In [ ]:

print(f"Data shadata/processedpe after RDP vs original shape: {df_rdp.shape} vs {df.shape}")

df_reduced = drop_constant_columns(df_rdp)

df_subset = df_reduced[target_features]

df_scaled, scalers = fit_preprocess_scalers(
    df_subset,
    normalize=True,
    standardize=False,
)

processed_data_path.parent.mkdir(parents=True, exist_ok=True)

In [29]:
df_scaled["mass"].shape
df_scaled["logT"].shape

(582166,)

In [ ]:
# time = df_scaled['star_age'].to_numpy()[::10]
# mass = df_scaled['mass'].to_numpy()[::10]
# temp = df_scaled['logT'].to_numpy()[::10]

# fig = plt.figure(figsize=(10, 7))
# ax = fig.add_subplot(projection='3d')

# ax.plot_trisurf(time, mass, temp, cmap='viridis', edgecolor='none')
# ax.set_xlabel('Star Age')
# ax.set_ylabel('Mass')
# ax.set_zlabel('Log T')
# ax.set_title('Stellar Evolution Surface')
# plt.show()

KeyError: 'star_age'

In [ ]:
fig = go.Figure(
    data=[
        go.Mesh3d(
            x=time,
            y=mass,
            z=temp,
            intensity=temp,
            colorscale="Viridis",
            opacity=0.8,
        )
    ]
)
fig.update_layout(
    scene=dict(
        xaxis_title="Star Age",
        yaxis_title="Mass",
        zaxis_title="Log T",
    ),
    title="Stellar Evolution Surface",
)
fig.show()

In [4]:
# importlib.reload(sys.modules["src.train"])
from src.train import train_model

In [5]:
metrics = train_model()

Training on device: cuda
Model: VAE(
  (encoder): Sequential(
    (0): Linear(in_features=2, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2)
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): LeakyReLU(negative_slope=0.2)
  )
  (fc_mu): Linear(in_features=128, out_features=4, bias=True)
  (fc_logvar): Linear(in_features=128, out_features=4, bias=True)
  (decoder): Sequential(
    (0): Linear(in_features=4, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2)
    (3): Linear(in_features=128, out_features=256, bias=True)
    (4): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): LeakyRe

/work/nvme/bhvr/cadence/Stars_VAE/src/train.py:15: UserWarning: Using a target size (torch.Size([32, 2])) that is different to the input size (torch.Size([32, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  mse = F.mse_loss(reconstructed, original, reduction="sum")
/work/nvme/bhvr/cadence/Stars_VAE/src/train.py:15: UserWarning: Using a target size (torch.Size([4, 2])) that is different to the input size (torch.Size([4, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  mse = F.mse_loss(reconstructed, original, reduction="sum")
/work/nvme/bhvr/cadence/Stars_VAE/src/train.py:15: UserWarning: Using a target size (torch.Size([28, 2])) that is different to the input size (torch.Size([28, 1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  mse = F.mse_loss(reconstructed, original, reduction="sum")


Epoch 0 | Train MSE: 0.1929 | Val MSE: 0.1905 | Gap: 0.0024
Epoch 1 | Train MSE: 0.1901 | Val MSE: 0.1902 | Gap: 0.0001
Epoch 2 | Train MSE: 0.1898 | Val MSE: 0.1897 | Gap: 0.0001
Epoch 3 | Train MSE: 0.1897 | Val MSE: 0.1895 | Gap: 0.0002
Epoch 4 | Train MSE: 0.1895 | Val MSE: 0.1894 | Gap: 0.0001
Epoch 5 | Train MSE: 0.1895 | Val MSE: 0.1900 | Gap: 0.0005
Epoch 6 | Train MSE: 0.1894 | Val MSE: 0.1893 | Gap: 0.0001
Epoch 7 | Train MSE: 0.1894 | Val MSE: 0.1892 | Gap: 0.0001


KeyboardInterrupt: 

In [ ]:
epochs = range(len(metrics["train_mse"]))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

ax1.plot(epochs, metrics["train_mse"], label="Train MSE")
ax1.plot(epochs, metrics["val_mse"], label="Validation MSE")
ax1.set_title("Reconstruction Loss (MSE)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("MSE")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.6)

ax2.plot(epochs, metrics["train_kld"], label="Train KLD")
ax2.plot(epochs, metrics["val_kld"], label="Validation KLD")
ax2.set_title("Latent Divergence (KLD)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("KLD")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()